In [ ]:
import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from collections import Counter
from tqdm import tqdm

os.makedirs("Phase9_Results", exist_ok=True)
os.makedirs("Phase9_Results/figures", exist_ok=True)

with open("Phase4_Results/phase4_results.json") as f:
    p3_records = json.load(f)

with open("Phase7_Results/spatial_metrics.json") as f:
    p7_records = json.load(f)

with open("Phase8_Results/phase8_labels.json") as f:
    p8_records = json.load(f)

print(f"Phase 3/5 records:  {len(p3_records)}")
print(f"Phase 7 records:    {len(p7_records)}")
print(f"Phase 8 records:    {len(p8_records)}")

p3_hall = sum(1 for r in p3_records if r["hallucinated"])
p8_hall = sum(1 for r in p8_records if r["hallucinated"])
print(f"\nPhase 3 hallucinated: {p3_hall}/7000 ({p3_hall/70:.1f}%)")
print(f"Phase 8 hallucinated: {p8_hall}/10000 ({p8_hall/100:.1f}%)")

y_p3       = np.array([1 if r["hallucinated"] else 0 for r in p3_records])
X_var_p3   = np.array([r["variance"]   for r in p3_records])
X_clip_p3  = np.array([r["clip_score"] for r in p3_records])

p7_lookup = {r["key"]: r for r in p7_records}
X_sci_p3  = np.array([p7_lookup[r["key"]]["sci_gini"]       for r in p3_records])
X_pvr_p3  = np.array([p7_lookup[r["key"]]["peak_var_ratio"] for r in p3_records])
X_smax_p3 = np.array([p7_lookup[r["key"]]["spatial_max"]    for r in p3_records])
X_sstd_p3 = np.array([p7_lookup[r["key"]]["spatial_std"]    for r in p3_records])

y_p8       = np.array([1 if r["hallucinated"] else 0 for r in p8_records])
X_var_p8   = np.array([r["variance"]   for r in p8_records])
X_clip_p8  = np.array([r["clip_score"] for r in p8_records])

print("\nArrays built successfully.")
print(f"Phase 3 shape: y={y_p3.shape}, X_var={X_var_p3.shape}, X_sci={X_sci_p3.shape}")
print(f"Phase 8 shape: y={y_p8.shape}, X_var={X_var_p8.shape}")

Phase 3/5 records:  7000
Phase 7 records:    7000
Phase 8 records:    10000

Phase 3 hallucinated: 574/7000 (8.2%)
Phase 8 hallucinated: 97/10000 (1.0%)

Arrays built successfully.
Phase 3 shape: y=(7000,), X_var=(7000,), X_sci=(7000,)
Phase 8 shape: y=(10000,), X_var=(10000,)


In [ ]:
def bootstrap_auc(y, X_signal, negate=False, n_boot=1000, seed=42,
                  ci_level=0.95, combined_X=None, is_combined=False):
    """
    Compute bootstrap confidence interval for ROC-AUC.
    
    Parameters:
        y          : binary labels (0/1)
        X_signal   : feature array (1D for single signal, 2D for combined)
        negate     : if True, use -X (lower score = more hallucinated)
        n_boot     : number of bootstrap iterations
        seed       : random seed for reproducibility
        ci_level   : confidence level (0.95 = 95% CI)
        combined_X : if provided, fit LR on this and compute AUC
        is_combined: if True, X_signal is already a 2D feature matrix
    
    Returns dict with: point_estimate, ci_lower, ci_upper, ci_width, boot_aucs
    """
    rng = np.random.default_rng(seed)
    n   = len(y)
    boot_aucs = []

    for _ in range(n_boot):
        # Sample with replacement
        idx = rng.integers(0, n, size=n)
        y_b = y[idx]

        
        if y_b.sum() == 0 or y_b.sum() == n:
            continue

        if is_combined:
            X_b = X_signal[idx]
            # Fit logistic regression on bootstrap sample
            sc  = StandardScaler()
            X_b_sc = sc.fit_transform(X_b)
            try:
                lr = LogisticRegression(random_state=42, max_iter=1000)
                lr.fit(X_b_sc, y_b)
                proba = lr.predict_proba(X_b_sc)[:,1]
                auc   = roc_auc_score(y_b, proba)
            except Exception:
                continue
        else:
            X_b = X_signal[idx]
            X_use = -X_b if negate else X_b
            try:
                auc = roc_auc_score(y_b, X_use)
            except Exception:
                continue

        boot_aucs.append(auc)

    boot_aucs = np.array(boot_aucs)
    alpha = (1 - ci_level) / 2
    ci_lo = np.percentile(boot_aucs, alpha * 100)
    ci_hi = np.percentile(boot_aucs, (1 - alpha) * 100)

    # Point estimate on full data
    if is_combined:
        sc_full = StandardScaler()
        X_sc    = sc_full.fit_transform(X_signal)
        lr_full = LogisticRegression(random_state=42, max_iter=1000)
        lr_full.fit(X_sc, y)
        proba_full = lr_full.predict_proba(X_sc)[:,1]
        point = roc_auc_score(y, proba_full)
    else:
        X_use = -X_signal if negate else X_signal
        point = roc_auc_score(y, X_use)

    return {
        "point_estimate": float(point),
        "ci_lower"      : float(ci_lo),
        "ci_upper"      : float(ci_hi),
        "ci_width"      : float(ci_hi - ci_lo),
        "n_valid_boots" : len(boot_aucs),
        "boot_aucs"     : boot_aucs.tolist(),
    }

print("Bootstrap function defined.")
print("Settings: n_boot=1000, CI=95%, seed=42")
print("Estimated time: ~3-5 minutes for all signals")

Bootstrap function defined.
Settings: n_boot=1000, CI=95%, seed=42
Estimated time: ~3-5 minutes for all signals


In [ ]:
X_p3_comb5 = np.column_stack([X_var_p3, X_clip_p3])
X_p3_combAll = np.column_stack([X_var_p3, X_clip_p3, 
                                  X_sci_p3, X_pvr_p3, X_smax_p3, X_sstd_p3])
X_p8_comb   = np.column_stack([X_var_p8, X_clip_p8])

N_BOOT = 1000
results_ci = {}

signals_to_boot = [
    # (label, y, X, negate, is_combined)
    ("Phase3_Variance",       y_p3, X_var_p3,    False, False),
    ("Phase3_CLIP",           y_p3, X_clip_p3,   True,  False),
    ("Phase3_Combined_Ph5",   y_p3, X_p3_comb5,  False, True),
    ("Phase7_SCI",            y_p3, X_sci_p3,    True,  False),
    ("Phase7_PVR",            y_p3, X_pvr_p3,    True,  False),
    ("Phase7_Combined_All",   y_p3, X_p3_combAll,False, True),
    ("Phase8_Variance",       y_p8, X_var_p8,    False, False),
    ("Phase8_CLIP",           y_p8, X_clip_p8,   True,  False),
    ("Phase8_Combined",       y_p8, X_p8_comb,   False, True),
]

for label, y, X, negate, is_comb in tqdm(signals_to_boot, desc="Bootstrapping"):
    results_ci[label] = bootstrap_auc(
        y=y, X_signal=X, negate=negate,
        n_boot=N_BOOT, seed=42, is_combined=is_comb
    )
    r = results_ci[label]
    print(f"  {label:<30} AUC={r['point_estimate']:.4f}  "
          f"95% CI [{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]  "
          f"width={r['ci_width']:.4f}")

print(f"\nAll bootstraps complete.")

Bootstrapping:  11%|█         | 1/9 [00:02<00:16,  2.00s/it]

  Phase3_Variance                AUC=0.8766  95% CI [0.8686, 0.8843]  width=0.0157


Bootstrapping:  22%|██▏       | 2/9 [00:04<00:14,  2.00s/it]

  Phase3_CLIP                    AUC=0.8725  95% CI [0.8576, 0.8875]  width=0.0299


Bootstrapping:  33%|███▎      | 3/9 [00:13<00:32,  5.50s/it]

...

  Phase8_Variance                AUC=0.9585  95% CI [0.9520, 0.9640]  width=0.0120


Bootstrapping:  89%|████████▉ | 8/9 [00:47<00:06,  6.28s/it]

  Phase8_CLIP                    AUC=0.4751  95% CI [0.4186, 0.5328]  width=0.1141


Bootstrapping: 100%|██████████| 9/9 [00:59<00:00,  6.66s/it]

  Phase8_Combined                AUC=0.9604  95% CI [0.9514, 0.9668]  width=0.0154

All bootstraps complete.


In [4]:
print("=" * 75)
print("PHASE 9 — BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 75)
print(f"{'Signal':<30} {'AUC':>7} {'CI Lower':>10} {'CI Upper':>10} "
      f"{'Width':>8} {'Interpretation'}")
print("─" * 75)

display_order = [
    ("Global Variance (Ph3)",     "Phase3_Variance",     "Phase 5 baseline"),
    ("CLIP Score (Ph3)",          "Phase3_CLIP",         "Phase 5 baseline"),
    ("Combined Ph5 (Var+CLIP)",   "Phase3_Combined_Ph5", "Phase 5 combined"),
    ("SCI / Gini (Ph7)",          "Phase7_SCI",          "Phase 7 spatial"),
    ("PVR (Ph7)",                 "Phase7_PVR",          "Phase 7 spatial"),
    ("All Features (Ph5+Ph7)",    "Phase7_Combined_All", "Phase 7 combined"),
    ("Variance External (Ph8)",   "Phase8_Variance",     "Phase 8 external"),
    ("CLIP External (Ph8)",       "Phase8_CLIP",         "Phase 8 external"),
    ("Combined External (Ph8)",   "Phase8_Combined",     "Phase 8 external"),
]

for display_name, key, interp in display_order:
    r = results_ci[key]
    print(f"  {display_name:<28} {r['point_estimate']:>7.4f} "
          f"{r['ci_lower']:>10.4f} {r['ci_upper']:>10.4f} "
          f"{r['ci_width']:>8.4f}  {interp}")

print()
print("Notes:")
print("  - All CIs computed with 1,000 bootstrap iterations, seed=42")
print("  - Percentile method (2.5th and 97.5th percentile)")
print("  - Combined signals use Logistic Regression refitted on each bootstrap sample")

PHASE 9 — BOOTSTRAP 95% CONFIDENCE INTERVALS
Signal                             AUC   CI Lower   CI Upper    Width Interpretation
───────────────────────────────────────────────────────────────────────────
  Global Variance (Ph3)         0.8766     0.8686     0.8843   0.0157  Phase 5 baseline
  CLIP Score (Ph3)              0.8725     0.8576     0.8875   0.0299  Phase 5 baseline
  Combined Ph5 (Var+CLIP)       0.8979     0.8862     0.9099   0.0237  Phase 5 combined
  SCI / Gini (Ph7)              0.8242     0.8093     0.8408   0.0315  Phase 7 spatial
  PVR (Ph7)                     0.8260     0.8112     0.8424   0.0313  Phase 7 spatial
  All Features (Ph5+Ph7)        0.9258     0.9175     0.9353   0.0177  Phase 7 combined
  Variance External (Ph8)       0.9585     0.9520     0.9640   0.0120  Phase 8 external
  CLIP External (Ph8)           0.4751     0.4186     0.5328   0.1141  Phase 8 external
  Combined External (Ph8)       0.9604     0.9514     0.9668   0.0154  Phase 8 external

Not

In [ ]:
boot_distributions = {k: v.pop("boot_aucs") for k, v in results_ci.items()}

with open("Phase9_Results/bootstrap_ci.json", "w") as f:
    json.dump(results_ci, f, indent=2)

print("Saved: Phase9_Results/bootstrap_ci.json")
print(json.dumps(results_ci, indent=2))

Saved: Phase9_Results/bootstrap_ci.json
{
  "Phase3_Variance": {
    "point_estimate": 0.8765828824754834,
    "ci_lower": 0.868587178188311,
    "ci_upper": 0.8843284980514503,
    "ci_width": 0.015741319863139314,
    "n_valid_boots": 1000
  },
  "Phase3_CLIP": {
    "point_estimate": 0.8724684995949599,
    "ci_lower": 0.8576022383382867,
    "ci_upper": 0.8874689529170032,
    "ci_width": 0.02986671457871648,
    "n_valid_boots": 1000
  },
  "Phase3_Combined_Ph5": {
    "point_estimate": 0.8978637525470893,
    "ci_lower": 0.8862482732395695,
    "ci_upper": 0.9099383416834551,
    "ci_width": 0.02369006844388566,
    "n_valid_boots": 1000
  },
  "Phase7_SCI": {
    "point_estimate": 0.8241836029804875,
    "ci_lower": 0.8092767105798762,
    "ci_upper": 0.8407833081360259,
    "ci_width": 0.03150659755614971,
    "n_valid_boots": 1000
  },
  "Phase7_PVR": {
    "point_estimate": 0.8259675143770244,
    "ci_lower": 0.8111590737145542,
    "ci_upper": 0.8424309426093638,
    "ci_wid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

internal_signals = [
    ("Variance\n(Ph3)",      "Phase3_Variance",    '#ff4444'),
    ("CLIP\n(Ph3)",          "Phase3_CLIP",        '#4488ff'),
    ("Combined\nPh5",        "Phase3_Combined_Ph5",'#aa44ff'),
    ("SCI\n(Ph7)",           "Phase7_SCI",         '#ff9900'),
    ("PVR\n(Ph7)",           "Phase7_PVR",         '#ffcc00'),
    ("All Features\nPh5+Ph7","Phase7_Combined_All",'#44cc88'),
]

x_int = np.arange(len(internal_signals))
for i, (label, key, color) in enumerate(internal_signals):
    r   = results_ci[key]
    pt  = r['point_estimate']
    lo  = r['ci_lower']
    hi  = r['ci_upper']
    axes[0].bar(i, pt, color=color, alpha=0.80, width=0.6, zorder=3)
    axes[0].errorbar(i, pt, yerr=[[pt-lo],[hi-pt]],
                     fmt='none', color='white', capsize=6,
                     capthick=2, elinewidth=2, zorder=4)
    axes[0].text(i, hi + 0.004, f'{pt:.3f}', ha='center', va='bottom',
                 color='white', fontsize=8, fontweight='bold')
    axes[0].text(i, lo - 0.012,
                 f'[{lo:.3f},\n{hi:.3f}]',
                 ha='center', va='top', color='#aaaaaa', fontsize=6.5)

axes[0].axhline(0.90, color='white',    lw=1, ls=':', alpha=0.5, label='AUC=0.90')
axes[0].axhline(0.80, color='#888888',  lw=1, ls=':', alpha=0.5, label='AUC=0.80')
axes[0].axhline(0.50, color='#555555',  lw=0.8, ls=':', alpha=0.4, label='Random')
axes[0].set_xticks(x_int)
axes[0].set_xticklabels([s[0] for s in internal_signals], color='white', fontsize=9)
axes[0].set_ylim(0.35, 1.02)
axes[0].set_ylabel('ROC-AUC', color='white', fontsize=11)
axes[0].set_title('Internal Signals — Phase 3/5/7\nError bars = 95% Bootstrap CI (n=1,000)',
                  color='white', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')

external_signals = [
    ("Variance\nInternal", "Phase3_Variance",    '#4488ff', 'Internal'),
    ("Variance\nExternal", "Phase8_Variance",    '#44cc88', 'External'),
    ("CLIP\nInternal",     "Phase3_CLIP",        '#aa44ff', 'Internal'),
    ("CLIP\nExternal",     "Phase8_CLIP",        '#ff4444', 'External'),
    ("Combined\nInternal", "Phase3_Combined_Ph5",'#4488ff', 'Internal'),
    ("Combined\nExternal", "Phase8_Combined",    '#44cc88', 'External'),
]

x_ext = np.arange(len(external_signals))
for i, (label, key, color, _) in enumerate(external_signals):
    r   = results_ci[key]
    pt  = r['point_estimate']
    lo  = r['ci_lower']
    hi  = r['ci_upper']
    axes[1].bar(i, pt, color=color, alpha=0.80, width=0.6, zorder=3)
    axes[1].errorbar(i, pt, yerr=[[pt-lo],[hi-pt]],
                     fmt='none', color='white', capsize=6,
                     capthick=2, elinewidth=2, zorder=4)
    axes[1].text(i, hi + 0.004, f'{pt:.3f}', ha='center', va='bottom',
                 color='white', fontsize=8, fontweight='bold')
    axes[1].text(i, lo - 0.014,
                 f'[{lo:.3f},\n{hi:.3f}]',
                 ha='center', va='top', color='#aaaaaa', fontsize=6.5)

axes[1].axvspan(-0.5,  1.5, alpha=0.05, color='#4488ff', label='Variance pair')
axes[1].axvspan( 1.5,  3.5, alpha=0.05, color='#aa44ff', label='CLIP pair')
axes[1].axvspan( 3.5,  5.5, alpha=0.05, color='#44cc88', label='Combined pair')
axes[1].axhline(0.50, color='#555555', lw=0.8, ls=':', alpha=0.5)
axes[1].axhline(0.90, color='white',   lw=1,   ls=':', alpha=0.4)
axes[1].set_xticks(x_ext)
axes[1].set_xticklabels([s[0] for s in external_signals], color='white', fontsize=9)
axes[1].set_ylim(0.25, 1.05)
axes[1].set_ylabel('ROC-AUC', color='white', fontsize=11)
axes[1].set_title('Internal vs External — Phase 3 vs Phase 8\nCI overlap confirms statistical robustness',
                  color='white', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')

fig.suptitle(
    'Phase 9 — Bootstrap 95% Confidence Intervals on All AUC Estimates\n'
    'Variance external CI [0.952, 0.964] does not overlap with CLIP external CI [0.419, 0.533]',
    color='white', fontsize=13, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig("Phase9_Results/figures/figure10_bootstrap_ci.png",
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.close()
print("Figure 10 saved → Phase9_Results/figures/figure10_bootstrap_ci.png")

Figure 10 saved → Phase9_Results/figures/figure10_bootstrap_ci.png


In [ ]:
print("=" * 60)
print("PHASE 9 COMPLETION CHECKLIST")
print("=" * 60)

checks = [
    ("Bootstrap CI computed for all 9 signals",
     len(results_ci) == 9),
    ("All 1,000 bootstrap iterations completed",
     all(v['n_valid_boots'] >= 950 for v in results_ci.values())),
    ("bootstrap_ci.json saved",
     os.path.exists("Phase9_Results/bootstrap_ci.json")),
    ("Figure 10 saved",
     os.path.exists("Phase9_Results/figures/figure10_bootstrap_ci.png")),
]

all_pass = True
for label, result in checks:
    status = "✓" if result else "✗ FAIL"
    if not result: all_pass = False
    print(f"  [{status}] {label}")

print()
print("KEY CI RESULTS:")
print(f"  Variance (internal):   {results_ci['Phase3_Variance']['point_estimate']:.4f}  "
      f"[{results_ci['Phase3_Variance']['ci_lower']:.4f}, "
      f"{results_ci['Phase3_Variance']['ci_upper']:.4f}]")
print(f"  Combined Ph5+Ph7:      {results_ci['Phase7_Combined_All']['point_estimate']:.4f}  "
      f"[{results_ci['Phase7_Combined_All']['ci_lower']:.4f}, "
      f"{results_ci['Phase7_Combined_All']['ci_upper']:.4f}]")
print(f"  Variance (external):   {results_ci['Phase8_Variance']['point_estimate']:.4f}  "
      f"[{results_ci['Phase8_Variance']['ci_lower']:.4f}, "
      f"{results_ci['Phase8_Variance']['ci_upper']:.4f}]")
print(f"  CLIP (external):       {results_ci['Phase8_CLIP']['point_estimate']:.4f}  "
      f"[{results_ci['Phase8_CLIP']['ci_lower']:.4f}, "
      f"{results_ci['Phase8_CLIP']['ci_upper']:.4f}]")
print()

var_ext_hi  = results_ci['Phase8_Variance']['ci_upper']
clip_ext_hi = results_ci['Phase8_CLIP']['ci_upper']
overlap = clip_ext_hi > results_ci['Phase8_Variance']['ci_lower']
print(f"  Variance vs CLIP (external) CIs overlap: {overlap}")
print(f"  → {'CIs do NOT overlap — statistically distinct signals' if not overlap else 'CIs overlap — signals not statistically distinct'}")

print()
if all_pass:
    print("✓ PHASE 9 COMPLETE")
else:
    print("✗ Some checks failed")

PHASE 9 COMPLETION CHECKLIST
  [✓] Bootstrap CI computed for all 9 signals
  [✓] All 1,000 bootstrap iterations completed
  [✓] bootstrap_ci.json saved
  [✓] Figure 10 saved

KEY CI RESULTS:
  Variance (internal):   0.8766  [0.8686, 0.8843]
  Combined Ph5+Ph7:      0.9258  [0.9175, 0.9353]
  Variance (external):   0.9585  [0.9520, 0.9640]
  CLIP (external):       0.4751  [0.4186, 0.5328]

  Variance vs CLIP (external) CIs overlap: False
  → CIs do NOT overlap — statistically distinct signals

✓ PHASE 9 COMPLETE


In [ ]:
clip_ci_lo = results_ci['Phase8_CLIP']['ci_lower']
clip_ci_hi = results_ci['Phase8_CLIP']['ci_upper']
random_in_ci = clip_ci_lo <= 0.50 <= clip_ci_hi

print(f"CLIP External CI: [{clip_ci_lo:.4f}, {clip_ci_hi:.4f}]")
print(f"0.50 (random) falls inside CI: {random_in_ci}")
print()
if random_in_ci:
    print("CORRECT STATEMENT:")
    print("CLIP external AUC (0.475) is not statistically distinguishable")
    print("from random chance — CI [0.419, 0.533] includes 0.50.")
    print("However, it is statistically distinct from variance external")
    print("[0.952, 0.964] — the CIs do not overlap.")
else:
    print("CLIP is statistically below random at 95% CI.")

# Show CFG breakdown of hallucinated images in Phase 5
from collections import Counter
cfg_hall = Counter(str(r['cfg']) for r in p3_records if r['hallucinated'])
print(f"\nPhase 5 hallucinated images by CFG:")
for cfg, count in sorted(cfg_hall.items(), key=lambda x: float(x[0])):
    pct = count/574*100
    print(f"  CFG={cfg}: {count} ({pct:.1f}% of all hallucinated)")

CLIP External CI: [0.4186, 0.5328]
0.50 (random) falls inside CI: True

CORRECT STATEMENT:
CLIP external AUC (0.475) is not statistically distinguishable
from random chance — CI [0.419, 0.533] includes 0.50.
However, it is statistically distinct from variance external
[0.952, 0.964] — the CIs do not overlap.

Phase 5 hallucinated images by CFG:
  CFG=0: 352 (61.3% of all hallucinated)
  CFG=1: 196 (34.1% of all hallucinated)
  CFG=3: 9 (1.6% of all hallucinated)
  CFG=5: 1 (0.2% of all hallucinated)
  CFG=10: 3 (0.5% of all hallucinated)
  CFG=15: 13 (2.3% of all hallucinated)


In [1]:


print()
print("1. PRIMARY EXTERNAL RESULT (Phase 8, CFG=7.5, 200 unseen prompts):")
print(f"   Variance AUC = 0.9585  CI = [0.9520, 0.9640]")
print(f"   CLIP AUC     = 0.4751  CI = [0.4186, 0.5328]")
print(f"   → CI [0.419, 0.533] includes 0.50: CLIP is NOT statistically")
print(f"     distinguishable from random chance on diverse external prompts.")
print(f"   → CIs do NOT overlap: variance and CLIP are statistically distinct.")
print(f"   Combined AUC = 0.9604  CI = [0.9514, 0.9668]")

print()
print("2. INTERNAL RESULT (Phase 5, CFG=0–15, 7,000 images):")
print(f"   Variance AUC = 0.8766  CI = [0.8686, 0.8843]")
print(f"   CLIP AUC     = 0.8725  CI = [0.8576, 0.8875]")
print(f"   Combined AUC = 0.8979  CI = [0.8862, 0.9099]")
print(f"   NOTE: 95.5% of hallucinated images (548/574) come from CFG=0 & CFG=1.")
print(f"   The pooled 0.877 reflects the full CFG range, not just production settings.")

print()
print("3. PER-CFG AUC ANALYSIS (computed from phase4_results.json):")
cfg_data = [
    (0.0,  352, 0.7708, 0.7613),
    (1.0,  196, 0.7595, 0.5216),
    (3.0,    9, 0.9058, 0.6955),
    (5.0,    1, None,   None),
    (7.5,    0, None,   None),
    (10.0,   3, 0.9097, 0.6369),
    (15.0,  13, 0.8490, 0.5167),
]
print(f"   {'CFG':<6} {'n_hall':>7} {'Var AUC':>9} {'CLIP AUC':>10}  Note")
print("   " + "-" * 48)
for cfg, nh, vauc, cauc in cfg_data:
    vs = f"{vauc:.4f}" if vauc else "undef"
    cs = f"{cauc:.4f}" if cauc else "undef"
    note = "★ CLIP near-random" if cauc and cauc < 0.56 and nh > 10 else ""
    print(f"   {cfg:<6} {nh:>7} {vs:>9} {cs:>10}  {note}")
print(f"   At CFG=1: variance (0.760) >> CLIP (0.522≈random)")
print(f"   → Variance discriminates where CLIP cannot, even at low guidance")

print()
print("4. SPATIAL TAXONOMY (Phase 7):")
print(f"   Hallucinated (n=574):   SCI mean = 0.386  (LOWEST — uniform spread)")
print(f"   Geometric only (n=364): SCI mean = 0.486  (HIGHEST — localised hot-spot)")
print(f"   Semantic only (n=1698): SCI mean = 0.410")
print(f"   Clean hall==F (n=6426): SCI mean = 0.436  (used in all stats)")
print(f"   Clean none    (n=4364): SCI mean = 0.442  (purer group, reference only)")
print(f"   Cohen's d (SCI): -1.014 — largest effect size in the entire study")
print(f"   All-features AUC: 0.926  CI = [0.918, 0.935]")

print()
print("5. CFG U-CURVE (Phase 5):")
print(f"   Descending arm (CFG 0→7.5): rho = -0.624, p ≈ 0  (strong)")
print(f"   Ascending arm  (CFG 7.5→15): rho = +0.144, p = 2.23e-15")
print(f"   NOTE: rho=+0.144 is statistically significant but weak.")
print(f"   The rebound is real but modest — do not overstate the ascending arm.")

print()
print("6. KEY CORRECTIONS FROM AUDIT:")
print(f"   [FIXED] CLIP external is NOT 'below random' — CI includes 0.50")
print(f"   [FIXED] Correct claim: 'not statistically distinguishable from random'")
print(f"   [FIXED] Phase 7 clean SCI: 0.436 (n=6426, halluc==False) used in stats")
print(f"   [FIXED] Per-CFG AUC added to Phase 5 (Section 2B)")
print(f"   [NOTED] 9 MANUAL prompts, 11 AUTO prompts (report had 7/13)")
print(f"   [NOTED] Ascending arm rho=+0.144 is weak despite being significant")

print()
print("✓ PHASE 9 COMPLETE — All corrections integrated")



1. PRIMARY EXTERNAL RESULT (Phase 8, CFG=7.5, 200 unseen prompts):
   Variance AUC = 0.9585  CI = [0.9520, 0.9640]
   CLIP AUC     = 0.4751  CI = [0.4186, 0.5328]
   → CI [0.419, 0.533] includes 0.50: CLIP is NOT statistically
     distinguishable from random chance on diverse external prompts.
   → CIs do NOT overlap: variance and CLIP are statistically distinct.
   Combined AUC = 0.9604  CI = [0.9514, 0.9668]

2. INTERNAL RESULT (Phase 5, CFG=0–15, 7,000 images):
   Variance AUC = 0.8766  CI = [0.8686, 0.8843]
   CLIP AUC     = 0.8725  CI = [0.8576, 0.8875]
   Combined AUC = 0.8979  CI = [0.8862, 0.9099]
   NOTE: 95.5% of hallucinated images (548/574) come from CFG=0 & CFG=1.
   The pooled 0.877 reflects the full CFG range, not just production settings.

3. PER-CFG AUC ANALYSIS (computed from phase4_results.json):
   CFG     n_hall   Var AUC   CLIP AUC  Note
   ------------------------------------------------
   0.0        352    0.7708     0.7613  
   1.0        196    0.7595     0